# 01 — Ingest & retrieve (Phase 1)

**Goal of this notebook:** take a handful of source PDFs, chunk them, embed the
chunks locally, store them in Chroma, and show retrieval working end to end —
ask a question, get back the most relevant passages *with citations* (document
title + page number). No LLM and no API yet; Phase 1 is about proving the
retrieval half is sound before anything is wrapped in a service.

**How the code is organised.** The notebook orchestrates; the logic lives in
three small modules under `src/rag_tutoring/`, installed as an editable package:

| module | responsibility |
|---|---|
| `config.py` | paths + the few tunable knobs (embedding model, token budget) |
| `ingest.py` | PDF → page-tagged, overlapping text chunks |
| `store.py`  | `VectorStore`: embed + store + query, behind a narrow interface |

That last seam is deliberate (see `docs/decisions/0001`): nothing here calls
Chroma directly, so swapping the store later touches one file.

> ⚠️ **Before committing this notebook, clear its outputs.** The retrieval cells
> print verbatim passages from copyrighted source PDFs — the same text `data/`
> is git-ignored to keep out of the repo. The closing cell has the one-liner.

In [ ]:
%load_ext autoreload
%autoreload 2

from rag_tutoring import config
from rag_tutoring.ingest import chunk_pdf
from rag_tutoring.store import VectorStore

print("project root:   ", config.repo_root())
print("embedding model:", config.EMBEDDING_MODEL)
print("chunk budget:   ", config.CHUNK_MAX_TOKENS, "word-pieces,",
      config.CHUNK_OVERLAP_TOKENS, "overlap  (model limit:", config.MODEL_MAX_TOKENS, ")")

## 1. The starter slice

The full corpus is 37 documents. Iterating on chunking against all of them is
slow, so this notebook works a five-paper slice chosen to tell one coherent
story — the road to modern NLP:

1. **word2vec** — *Efficient Estimation of Word Representations in Vector Space*
2. **GloVe** — *Global Vectors for Word Representation*
3. **Transformer** — *Attention Is All You Need*
4. **BERT** — *Pre-training of Deep Bidirectional Transformers*
5. **RAG** — *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*

The RAG paper is in here on purpose: it lets the tutor explain *its own*
architecture, which is a nice thing to demo. Scaling to the full corpus is just
a longer file list — papers and textbooks share one chunk setting, for the
reason explained in §3.

In [ ]:
PAPERS = config.data_raw_dir() / "papers"
STARTER = [
    PAPERS / "Efficient Estimation of Word Representations in Vector Space.pdf",
    PAPERS / "GloVe- Global Vectors for Word Representation.pdf",
    PAPERS / "Attention Is All You Need.pdf",
    PAPERS / "BERT- Pre-training of Deep Bidirectional Transformers for Language Understanding.pdf",
    PAPERS / "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.pdf",
]

missing = [p.name for p in STARTER if not p.exists()]
assert not missing, f"missing starter PDFs: {missing}"
print(f"all {len(STARTER)} starter PDFs present")

## 2. Load & inspect

Confirm the text actually extracts before trusting any of it downstream. For
each paper we print the page count and a snippet, so a scanned or image-only
PDF (which would extract as near-empty) is obvious immediately.

`load_pdf` also strips lone surrogates — half of a surrogate pair left behind
when extraction mangles a symbol outside the BMP (usually a mathematical
alphanumeric character). They make the string invalid UTF-8, and the tokenizer
rejects it with a `TypeError` rather than anything self-explanatory. Only a
couple of textbook pages carry one, which is exactly why an all-papers slice
never surfaced it.

In [ ]:
from rag_tutoring.ingest import load_pdf

for path in STARTER:
    pages = load_pdf(path)
    total_chars = sum(len(t) for _, t in pages)
    print(f"{path.stem[:55]:55s}  {len(pages):3d} pages  {total_chars:>7,} chars")

# Spot-check one paper's first page of extracted text.
sample_pages = load_pdf(STARTER[2])  # Attention Is All You Need
print("\n--- sample: first page of 'Attention Is All You Need' ---\n")
print(sample_pages[0][1][:600])

## 3. Chunk

Split each page into overlapping windows, staying inside a page so every chunk
cites exactly one page.

**Windows are measured in word-pieces, not words.** The embedding model accepts
256 word-pieces and silently truncates anything longer — no error, no warning.
Words are the wrong unit for that budget: in this corpus a word costs anywhere
from 1 to ~4 word-pieces (math notation and long technical terms split hard), so
a fixed word count cannot bound the result. Measured on this slice, the previous
160-word setting produced chunks from 90 to **549** word-pieces, and 34% of them
overflowed the model's window.

The failure that causes is subtle and specific to RAG: the chunk text is stored
and *displayed as a citation*, but only the part that fit was embedded. The
passage shown to a student and the vector it was matched on drift apart.

So chunks are packed greedily to a token budget instead. Two details that matter:

- **Chunks keep their original words.** Rebuilding text by decoding token ids
  would be lossy — this tokenizer is uncased and splits punctuation, so
  `"Multi-Head Attention (Vaswani et al., 2017)"` comes back as
  `"multi - head attention ( vaswani et al., 2017 )"`. Citations must survive
  verbatim, so only the *cost* of each word comes from the tokenizer.
- **Papers and textbooks share one setting**, because the budget is imposed by
  the embedding model's input window, which does not care what kind of document
  the text came from.

In [ ]:
from rag_tutoring.ingest import token_counter

count_tokens = token_counter()  # built once and reused; memoised per word

chunks = []
for path in STARTER:
    doc_chunks = chunk_pdf(path, source_type="paper", count_tokens=count_tokens)
    chunks.extend(doc_chunks)
    print(f"{path.stem[:55]:55s}  {len(doc_chunks):4d} chunks")

print(f"\ntotal chunks: {len(chunks)}")

# What one chunk actually looks like: text + the metadata that makes it citable.
c = chunks[len(chunks) // 2]
print(
    f"\n--- example chunk ---"
    f"\nid:    {c.id}"
    f"\nsource:{c.source} (p.{c.page}, {c.source_type})"
    f"\ntext:  {c.text[:300]}..."
)

Check the budget actually holds before spending time on embeddings. This is the
assertion whose absence let the truncation bug sit unnoticed: nothing throws
when a chunk is too long, so the invariant has to be tested explicitly.

In [ ]:
sizes = [sum(count_tokens(w) for w in c.text.split()) + 2 for c in chunks]  # +2 = [CLS]/[SEP]
print(f"chunk size in word-pieces: max={max(sizes)}  mean={sum(sizes)/len(sizes):.1f}  "
      f"limit={config.MODEL_MAX_TOKENS}")

# strict=True because a length mismatch here would make the assertion below pass
# vacuously: plain zip() stops at the shorter sequence, so any unmeasured chunks
# would simply not be checked. Same trap as §6 — a check that cannot fail.
over = [c.id for c, n in zip(chunks, sizes, strict=True) if n > config.MODEL_MAX_TOKENS]
assert not over, f"{len(over)} chunks would be silently truncated: {over[:3]}"
print(f"OK — all {len(chunks)} chunks fit the embedding window, so nothing is truncated.")

## 4. Embed & index

`VectorStore` loads the local sentence-transformers model, embeds each chunk,
and upserts it into a persistent Chroma collection (under `chroma/`,
git-ignored). The collection is named after the embedding model — a different
model means a different, incomparable index, so it gets its own collection.

*First run downloads the model (~90 MB) and embeds the slice — expect a minute
or two. Re-runs are fast; upsert overwrites by id rather than duplicating.*

In [ ]:
store = VectorStore()
print("collection:", store.collection_name)

# reset() first so re-running after a chunk-size change rebuilds cleanly
# instead of leaving stale chunks from the previous run behind.
store.reset()
store.add(chunks)
print("indexed chunks:", store.count())

## 5. Retrieve

The payoff. Each question is embedded with the same model and matched against
the index. Results come back as `Retrieved` objects carrying the citation
(`source`, `page`) and a **cosine similarity** in `[0, 1]` — higher is closer.

In [ ]:
def show(question, k=4):
    print(f"Q: {question}\n" + "-" * 78)
    for i, hit in enumerate(store.query(question, k=k), start=1):
        snippet = " ".join(hit.text.split())[:220]
        print(f"{i}. [{hit.score:.3f}] {hit.source} (p.{hit.page})\n   {snippet}...\n")

show("What is retrieval-augmented generation and how does it use a retriever?")

In [ ]:
show("How does multi-head self-attention work in the Transformer?")

In [ ]:
show("What problem do learned word embeddings like word2vec solve?")

## 6. Sanity checks: does the index round-trip, and is the score really cosine?

Two separate claims, and it matters that they're tested separately.

**(a) Round-trip.** Query with the exact text of a known chunk; the top hit must
be that chunk at similarity ≈ 1.0. This proves the chunk was stored and is
retrievable — nothing more.

**(b) The metric.** Chroma's distance function comes from collection metadata,
and a silent fallback to L2 would leave the *ranking* correct (the embeddings
are normalised) while making the printed number meaningless.

The round-trip check is **not** evidence for (b), which is easy to get wrong:
for normalised vectors an L2 index also returns distance 0 on an exact match, so
it scores 1.0 too. Both metrics pass (a). Only a *non-identical* hit separates
them — cosine reports `1 - d = cos`, squared-L2 reports `1 - 2(1 - cos)`. So (b)
is tested by recomputing the similarity by hand and comparing.

*(Related trap: `get_or_create_collection` ignores the `metadata` argument when
the collection already exists. A collection first created under a different
metric would keep it forever, silently. `reset()` drops and recreates, which is
why re-indexing gets the metric it asked for.)*

In [ ]:
import numpy as np

# (a) round-trip
probe = chunks[len(chunks) // 2]
top = store.query(probe.text, k=1)[0]
assert top.text == probe.text, "identity query did not return the probe chunk"
assert top.score > 0.99, f"exact match should score ~1.0, got {top.score:.4f}"
print(f"(a) round-trip OK — {top.source[:44]} (p.{top.page}) scored {top.score:.4f}")

# (b) the metric, on NON-identical hits
question = "How does multi-head self-attention work in the Transformer?"
hits = store.query(question, k=3)
q_vec = store.model.encode([question], normalize_embeddings=True)[0]
h_vecs = store.model.encode([h.text for h in hits], normalize_embeddings=True)

print(f"\n{'reported':>10s} {'hand-computed':>14s}   source")
for hit, vec in zip(hits, h_vecs, strict=True):
    true_cos = float(np.dot(q_vec, vec))
    print(f"{hit.score:10.4f} {true_cos:14.4f}   {hit.source[:40]} (p.{hit.page})")
    assert abs(hit.score - true_cos) < 1e-4, (
        f"reported {hit.score:.4f} != cosine {true_cos:.4f} — collection is not using cosine"
    )
print("\n(b) OK — reported scores equal hand-computed cosine similarity.")

## Where this goes next

**Works today:** end-to-end retrieval with citations over the starter slice,
behind a store interface that isn't tied to Chroma.

**Known limitations (expected, not bugs):**
- These are two-column PDFs; pypdf sometimes interleaves columns, so a snippet
  can read slightly jumbled. It's extraction reading-order, not the pipeline.
- Per-page chunking splits any passage that straddles a page break.
- Textbook chunks will reference figures the retriever can't see ("the green
  shape") — fine as long as the surrounding prose carries the meaning.
- Non-content pages — title pages, acknowledgments, reference lists — are
  chunked and indexed like prose, and they compete with real content. The eval
  set measured this; see `eval/README.md`.

**Phase 1 is closed.** Both remaining steps above were finished outside this
notebook, because they need the whole corpus rather than the slice:

1. **Full corpus indexed** — 9,169 chunks from 37 documents, via
   `scripts/build_index.py` (a rebuild, so the index only ever holds chunks from
   the current setting) with `scripts/audit_corpus.py` as the regression net.
2. **Eval set built and scored** — 32 labelled questions plus 5 unanswerable
   ones, `recall@5 = 0.50`, broken out by how the question is phrased. Three
   queries looking right was never a measurement; this is. See `eval/README.md`.

Tuning the token budget and overlap comes next, and now it means something:
there is a number to move. Phase 2 is FastAPI + citation formatting.

---
**Before committing this notebook, clear outputs** (the cells above print
copyrighted source text):

```
jupyter nbconvert --to notebook --clear-output --inplace notebooks/01_ingest_and_retrieve.ipynb
```